In [ ]:
# Cell 1: Install Dependencies
!pip install -q transformers torch huggingface_hub

In [ ]:
# Cell 2: Setup Authentication & Imports
import os
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer

# Authenticate via Colab Secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
except Exception as e:
    print("Running without logged-in HF token:", e)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing on device: {device}")

In [ ]:
# ==============================================================================
# Cell 3: Token vs Token ID Mechanics (Using Ungated Qwen2.5)
# ==============================================================================
print("=== 1. Token vs Token ID Mechanics ===")
model_id_base = "Qwen/Qwen2.5-0.5B"

# Load base tokenizer
tokenizer_base = AutoTokenizer.from_pretrained(model_id_base)

sample_text = "I am excited to show Tokenizers in action to my LLM engineers."

# Encode: Text String -> Numerical Token IDs
token_ids = tokenizer_base.encode(sample_text)

print(f"Raw Input: '{sample_text}'")
print(f"Character Count: {len(sample_text)} | Word Count: {len(sample_text.split())}")
print(f"Token Count: {len(token_ids)}")
print(f"Token IDs (Integers): {token_ids}\n")

# Single Token Decoding vs Batch Subword Mapping
decoded_full = tokenizer_base.decode(token_ids)
subwords = tokenizer_base.batch_decode([[tid] for tid in token_ids])

print(f"Decoded Full Text: {decoded_full}")
print(f"Subword Token Chunks: {subwords}\n")

print(f"Vocabulary Size: {tokenizer_base.vocab_size}")
print(f"Total Entries (Vocab + Special Tokens): {len(tokenizer_base)}")

In [ ]:
# ==============================================================================
# Cell 4: Chat Templates (apply_chat_template)
# ==============================================================================
print("\n=== 2. Inspecting Chat Templates ===")
model_id_instruct = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer_instruct = AutoTokenizer.from_pretrained(model_id_instruct)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Tell a lighthearted joke for a room of data scientists."}
]

# Transform message dict list into single prompt string with special tokens
formatted_chat = tokenizer_instruct.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

print("Formatted Qwen 2.5 Chat String:")
print(formatted_chat)

In [ ]:
# ==============================================================================
# Cell 5: Cross-Architecture Comparisons (Fully Ungated)
# ==============================================================================
print("\n=== 3. Cross-Architecture Comparisons ===")
test_sentence = "I am curiously excited to show Hugging Face Tokenizers in action to my LLM engineers."

tok_qwen = tokenizer_instruct
tok_phi = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")
tok_qwen_coder = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-1.5B-Instruct")

models_dict = {
    "Qwen-2.5-0.5B": tok_qwen,
    "Phi-3.5-Mini": tok_phi,
    "Qwen-2.5-Coder": tok_qwen_coder
}

for name, tok in models_dict.items():
    ids = tok.encode(test_sentence)
    subword_list = tok.batch_decode([[i] for i in ids])
    print(f"\n--- {name} ---")
    print(f"Token Count: {len(ids)}")
    print(f"Token IDs (First 5): {ids[:5]}")
    print(f"Subwords (First 8): {subword_list[:8]} ...")

In [ ]:
# Cell 6: Specialized Code Tokenization
print("=== 4. Code Tokenization Breakdown ===")
python_code = "def hello_world(person):\n    print('Hello, ' + person)"

code_ids = tok_qwen_coder.encode(python_code)
code_tokens = tok_qwen_coder.batch_decode([[cid] for cid in code_ids])

print(f"Source Code:\n{python_code}\n")
print(f"Tokenized Code Sequence ({len(code_ids)} tokens):")
for tid, tstr in zip(code_ids, code_tokens):
    print(f"  ID: {tid:<6} | Token: {repr(tstr)}")